![Egeria Logo](https://raw.githubusercontent.com/odpi/egeria/main/assets/img/ODPi_Egeria_Logo_color.png)

### Coco Pharmaceuticals Labs

----

# Extending the systems inventory


This workbook demonstrates how to create a system inventory using Egeria.  The data to load has come from two sites that are not part of [Gary Geeke's initial spreadsheet](https://egeria-project.org/practices/coco-pharmaceuticals/scenarios/cataloguing-infrastructure/overview/).

<figure style="margin-left: 0%; display:inline-block;">
<img src="https://egeria-project.org/practices/coco-pharmaceuticals/scenarios/cataloguing-infrastructure/gary-geeke-updating-systems-spreadsheet.png">
<figcaption style="margin-left: 15%;"><strong>Gary Geeke working on the spreadsheet he used to manage system data before the switch to Egeria</strong></figcaption>
</figure>

The two new sites are from recently acquired pharmaceutical manufacturing companies.  Both have experties in small batch manufacturing which is a new capability required by Coco Pharmaceuticals to support their personalized medicine strategy.

Each site, one in Austin, Texas, USA and the other in Bucharest, Romania has sent Gary spreadsheets describing their systems.   Gary's job is to load the data in these spreadsheets into Egeria.

----

In [ ]:
# All CSV files live in the data directory
from pathlib import Path

DATA_DIR = Path('./data').resolve()

print(f"Data directory: {DATA_DIR}")
for f in sorted(DATA_DIR.glob('*.csv')):
    print(f"  {f.name}")

In [ ]:
# Helper functions to load CSV files
import csv

def load_csv(filename):
    """Return a list of dicts from a CSV file in DATA_DIR."""
    path = DATA_DIR / filename
    with open(path, newline='', encoding='utf-8') as fh:
        return list(csv.DictReader(fh))

def preview(rows, n=3):
    """Print the first n rows and a row count."""
    print(f"{len(rows)} rows loaded")
    for row in rows[:n]:
        for k, v in row.items():
            print(f"  {k}: {v}")
        print()

In [ ]:
# Initialize pyegeria

import os
view_server = os.environ.get("EGERIA_VIEW_SERVER","qs-view-server")
url = os.environ.get("EGERIA_VIEW_SERVER_URL","https://host.docker.internal:9443")
user_id = "garygeeke"
user_pwd = os.environ.get("EGERIA_USER_PASSWORD")
egeria_width = 150

print("\n")
print("Accessing view server " + view_server + " on platform " + url + " for user " + user_id)
print("\n")

# These packages support the different types of markdown display
from IPython.display import display, Markdown
from pyegeria import load_mermaid, render_mermaid
load_mermaid()

from datetime import datetime
import json
import time


# EgeriaTech combines many of the clients to call Egeria
from pyegeria import EgeriaTech

egeria_client = EgeriaTech(view_server, url, user_id, user_pwd)
token = egeria_client.create_egeria_bearer_token()



-----

The systems loaded from Gary's original spreadsheet are organised into subsystems under the **IT Systems Inventory**.  This can be seen from the **Egeria portal** by selecting **Egeria Explorer**, then **Collections**.  The systems from Austin and Bucharest will be added to the IT Systems Inventory.  Today they run independently so 

-----

In [ ]:
# Locate the IT Systems Inventory

token = egeria_client.create_egeria_bearer_token()

it_systems_inventory_name="RootCollection::Coco::IT Systems Inventory"
it_systems_inventory_guid=egeria_client.get_element_guid_by_unique_name(it_systems_inventory_name)

print ("IT Systems Inventory GUID: " + it_systems_inventory_guid)

----

## Austin Systems

The [coco_austin_systems.csv](tools/postgres-surveyor/data/coco_austin_systems.csv) file contains the 45 systems owned by the Austin location. Here's a breakdown of the systems by category:

| Category | Systems |
|---|---|
| **Hadoop data lake** | CDP primary & secondary clusters, edge node, Knox gateway, Spark, Kafka, Hive, NetApp storage |
| **ERP & Finance** | SAP S/4HANA + HANA DB, SAP EAM module |
| **Orders & Logistics** | Oracle Order Management, Manhattan WMS, Oracle TMS, OpenText EDI gateway |
| **Suppliers** | SAP Ariba SRM |
| **Customers & Marketing** | Salesforce CRM, Salesforce Marketing Cloud, ServiceNow Customer Service |
| **Employees & HR** | Workday HCM, UKG Time & Attendance, Cornerstone LMS |
| **Manufacturing & QC** | Siemens Opcenter MES, Rockwell SCADA, LabWare LIMS, Veeva QMS, Veeva EBR, Waters Empower chromatography, Honeywell environmental monitoring |
| **Regulatory** | Trackwise Digital |
| **Maintenance** | IBM Maximo CMMS |
| **IT Infrastructure** | Active Directory, Entra ID, Microsoft 365, ServiceNow ITSM, VMware vSphere, Veeam Backup |
| **Security** | Palo Alto firewall, Splunk SIEM, Microsoft Purview DLP |
| **Integration & Analytics** | Informatica IDMC (ETL), Informatica MDM, Power BI, Esri ArcGIS |
| **Networking** | Cisco Catalyst / Meraki |

The Hadoop data lake (AUS-SYS-001 through 008) sits at the centre with Kafka feeding real-time MES and SCADA data in, and Spark/Hive providing analytics on top. The Informatica ETL layer (AUS-SYS-042) ties the operational systems to the lake.

The systems are organized into subsystems.  Gary intends to load the subsystem data first so that the systems can be linked to their subsystems as they are created.  

Subsystems are represented as ITSubsystem collections. There is also a root subsystem for the Austin Data Center which is linked into the IT Systems Inventory.  

-----

In [ ]:
# Locate the Austin Location

token = egeria_client.create_egeria_bearer_token()

austin_dc_name="Data centre::AUS-DC"
austin_dc_guid=egeria_client.get_element_guid_by_unique_name(austin_dc_name)

print ("Austin Data Centre GUID: " + austin_dc_guid)


----

All subsystems and systems will be linked to the Austin Data Center's location.

----

In [ ]:
# Create the Austin Data Centre subsystem

token = egeria_client.create_egeria_bearer_token()

austin_subsystem_collection_name="Austin Data Centre Systems"
austin_subsystem_collection_desc="Systems located in the Austin data centre are self-contined at this moment and so are shown as a separate subsystem."
austin_subsystem_collection_qname="ITSubsystem::" + austin_subsystem_collection_name

body = {
    "class" : "NewElementRequestBody",
    "isOwnAnchor": True,
    "properties": {
        "class" : "ITSubsystemProperties",
        "qualifiedName" : austin_subsystem_collection_qname,
        "displayName" : austin_subsystem_collection_name,
        "description" : austin_subsystem_collection_desc,
        "category" : "Acquired Systems"
    },
    "parentGUID" : austin_dc_guid,
    "parentRelationshipTypeName" : "KnownLocation",
    "parentAtEnd1" : False
}


austin_tl_subsystem=egeria_client.get_element_by_unique_name(name=austin_subsystem_collection_qname)

if austin_tl_subsystem == "No elements found":
    austin_tl_subsystem_guid = egeria_client.create_collection(body=body)
    egeria_client.add_to_collection(collection_guid=it_systems_inventory_guid, element_guid=austin_tl_subsystem_guid)
else:
    austin_tl_subsystem_guid = austin_tl_subsystem.get("elementHeader").get("guid")
    
print ("Austin Top-level Subsystem GUID: " + austin_tl_subsystem_guid)



----

The data for these subsystems is in the following format:

-----

In [ ]:
# Load Austin Subsystems

austin_subsystems = load_csv('coco_austin_subsystems.csv')
preview(austin_subsystems)


----

This code loads the subsystem information into Egeria.  They are represented as ITSubsystem collections.  There is a root subsystem for the Austin Data Center which is linked into the **IT Systems Inventory**.

----

In [ ]:
# Add Austin Subsystems to Egeria

token = egeria_client.create_egeria_bearer_token()

subsys_map = dict()

for sub_sys in austin_subsystems:
    subsystem_id          = sub_sys['subsystem_id']
    subsystem_name        = sub_sys['subsystem_name']
    subsystem_description = sub_sys['subsystem_description']
    business_owner        = sub_sys['business_owner']
    member_system_ids     = sub_sys['member_system_ids']
   
    qualified_name = "ITSubsystem::" + subsystem_id + "::" + subsystem_name

    subsystem_guid=egeria_client.get_element_guid_by_unique_name(qualified_name)

    if subsystem_guid == "No elements found":
    
        body = {
            "class" : "NewElementRequestBody",
            "isOwnAnchor": True,
            "properties": {
                "class" : "ITSubsystemProperties",
                "qualifiedName" : qualified_name,
                "identifier" : subsystem_id,
                "displayName" : subsystem_name,
                "description" : subsystem_description,
                "category" : "Acquired Systems"
            },
            "parentGUID" : austin_dc_guid,
            "parentRelationshipTypeName" : "KnownLocation",
            "parentAtEnd1" : False
        }

        subsystem_guid = egeria_client.create_collection(body=body)
        egeria_client.add_to_collection(collection_guid=austin_tl_subsystem_guid, element_guid=subsystem_guid)

    subsys_map[subsystem_id] = subsystem_guid
    
    print(f"Processed: {subsystem_id}  {subsystem_description[:60]}...")
    #asset=egeria_client.get_asset_by_guid(system_guid)
    #print(json.dumps(asset,indent=2))
    
print(f"\nDone — {len(austin_subsystems)} Austin subsystems processed.")



----

This code loads the subsystem information into Egeria.  They are represented as ITSubsystem collections.  There is a root subsystem for the Austin Data Center which is linked into the **IT Systems Inventory**.

----

In [ ]:
# Load Austin systems

austin_systems = load_csv('coco_austin_systems.csv')
preview(austin_systems)


----

This code loads the system information into Egeria.  They are represented as SoftwareServer assets.

----

In [ ]:
# Add Austin Systems to Egeria

token = egeria_client.create_egeria_bearer_token()

sys_map = dict()

for sys in austin_systems:
    system_id            = sys['system_id']
    subsystem_id         = sys['subsystem_id']
    system_name          = sys['system_name']
    system_serial_number = sys['system_serial_number']
    system_install_date  = sys['system_install_date']
    description          = sys['description']
    operating_system     = sys['operating_system']
    latest_patch_level   = sys['latest_patch_level']
    latest_patch_date    = sys['latest_patch_date']

    zone_membership = ["business-systems"]

    qualified_name = "SoftwareServer::" + system_id + "::" + system_serial_number

    system_guid=egeria_client.get_element_guid_by_unique_name(qualified_name)

    if system_guid == "No elements found":
    
        body = {
            "class" : "NewElementRequestBody",
            "isOwnAnchor": True,
            "properties": {
                "class" : "SoftwareServerProperties",
                "qualifiedName" : qualified_name,
                "identifier" : system_serial_number,
                "displayName" : system_name,
                "resourceName" : system_id,
                "description" : description,
                "deployedImplementationType" : "Unclassified Software Server",
                "category" : "Acquired Systems",
                "deploymentStatus" : "ACTIVE",
                "additionalProperties" : {
                    "operatingSystem" : operating_system,
                    "latestPatchLevel" : latest_patch_level,
                    "latestPatchDate" : latest_patch_date
                }
            },
            "initialClassifications" : {
                "ZoneMembership" : {
                    "class" : "ZoneMembershipProperties",
                    "zoneMembership" : zone_membership
                }
            },
            "parentGUID" : austin_dc_guid,
            "parentRelationshipTypeName" : "KnownLocation",
            "parentAtEnd1" : False
        }

        system_guid = egeria_client.create_asset(body=body)
        egeria_client.add_to_collection(collection_guid=subsys_map[subsystem_id], element_guid=subsystem_guid)

    sys_map[system_id] = system_guid
    
    print(f"Processed: {system_id}  {description[:60]}...")
    #asset=egeria_client.get_asset_by_guid(system_guid)
    #print(json.dumps(asset,indent=2))
    
print(f"\nDone — {len(austin_systems)} Austin systems processed.")



----

### System integrations

The diagram below shows the interactions between all 45 systems:

* Shop floor → Kafka → Data Lake — MES, SCADA and environmental sensors stream events in real time through Kafka into the CDP clusters
* Quality systems → ETL → Data Lake — LIMS and chromatography results land in the lake via Informatica ETL alongside the ERP sync
* MDM as the master data hub — Informatica MDM pushes golden records for products, customers and suppliers out to SAP, Salesforce and the WMS
* SAP at the centre of finance/logistics — bi-directional flows to OMS, WMS, Ariba and Workday
* Entra ID providing SSO to all SaaS platforms (Salesforce, Workday, Veeva)
* Splunk collecting logs from the firewall and Active Directory for security monitoring

```mermaid
flowchart TB
subgraph SHOP["🏭 Shop Floor"]
MES["Siemens Opcenter\nMES"]
SCADA["Rockwell FactoryTalk\nSCADA"]
ENV["Honeywell\nEnv Monitor"]
CHR["Waters Empower\nChromatography"]
CMMS["IBM Maximo\nCMMS"] 
end 
subgraph QUALITY["🧪 Quality and Regulatory"]
LIMS["LabWare\nLIMS"] 
EBR["Veeva\nEBR"] 
QMS["Veeva\nQMS"] 
TRACK["Trackwise\nDigital"] 
end 
subgraph LAKE["🌊 Hadoop Data Lake"]
KAFKA["Apache\nKafka"] 
EDGE["CDP\nEdge Node"] 
CDP1["CDP Primary\nCluster"] 
CDP2["CDP Secondary\nCluster"] 
HIVE["Apache\nHive"] 
SPARK["Apache\nSpark"] 
NETAPP["NetApp\nStorage"] 
end 
subgraph INTEGRATION["🔄 Integration & Analytics"]
ETL["Informatica\nETL"] 
MDM["Informatica\nMDM"] 
BI["Power BI"] 
GIS["ArcGIS"] 
end 
subgraph ERP_SG["💰 ERP & Finance"]
SAP["SAP S/4HANA"] 
HANA["SAP HANA DB"] 
EAM["SAP EAM"] 
end 
subgraph SUPPLY["📦 Supply Chain & Logistics"]
OMS["Oracle\nOMS"] 
WMS["Manhattan\nWMS"] 
TMS["Oracle\nTMS"] 
EDI["OpenText\nEDI"] 
SRM["SAP Ariba\nSRM"] 
end 
subgraph CUSTOMERS["🤝 Customers & Marketing"]
CRM["Salesforce\nCRM"] 
MKT["Salesforce\nMarketing"] 
CSS["ServiceNow\nCSS"] 
end 
subgraph HR["👥 Human Resources"]
HCM["Workday\nHCM"] 
TNA["UKG\nTime & Attendance"] 
LMS["Cornerstone\nLMS"] 
end 
subgraph IT["🖥 IT Infrastructure"]
AD["Active\nDirectory"] 
AAD["Entra ID\n(Azure AD)"] 
M365["Microsoft\n365"] 
VMW["VMware\nvSphere"] 
BACKUP["Veeam\nBackup"] 
NET["Cisco\nNetwork"] 
ITSM["ServiceNow\nITSM"] 
end 
subgraph SEC["🔒 Security"]
FW["Palo Alto\nFirewall"] 
SIEM["Splunk\nSIEM"] 
DLP["MS Purview\nDLP"] 
end 
%% Shop floor --> Kafka (real-time streaming) 
MES -->|"events"| KAFKA
SCADA -->|"telemetry"| KAFKA
ENV -->|"sensor data"| KAFKA
%% Kafka --> Data Lake
KAFKA --> EDGE
EDGE --> CDP1
CDP1 <-->|"replication"| CDP2
CDP1 --- NETAPP
CDP1 --> HIVE
CDP1 --> SPARK
%% Quality --> ETL --> Data Lake
LIMS -->|"results"| ETL
CHR -->|"instrument data"| ETL
ETL --> EDGE
%% EBR <--> MES
EBR <-->|"batch records"| MES
%% QMS links 
QMS <-->|"deviations"| MES
QMS <-->|"sample events"| LIMS
QMS --> TRACK
%% ERP SAP --- HANA
SAP --- EAM
SAP <-->|"sync"| ETL
%% 
SAP --> EDGE
ETL --> EDGE
%% MDM feeds master data
MDM -->|"product data"| SAP
MDM -->|"customer data"| CRM
MDM -->|"supplier data"| SRM
MDM -->|"item data"| WMS
%% Supply chain 
OMS <-->|"fulfilment"| WMS
WMS <-->|"shipments"| TMS
TMS <-->|"routing"| EDI
SRM <-->|"orders"| EDI
SAP <-->|"PO / GR"| SRM
SAP <-->|"orders"| OMS
SAP <-->|"inventory"| WMS
%% Customers 
CRM -->|"orders"| OMS
CRM --- MKT 
CSS <-->|"complaints"| QMS 
%% HR 
HCM <-->|"payroll"| SAP 
HCM --- TNA
HCM --- LMS
%% Analytics 
HIVE -->|"lake data"| BI
SAP -->|"financials"| BI
TMS -->|"routes"| GIS
%% IT infrastructure 
AD -->|"auth"| VMW
AAD -->|"SSO/MFA"| CRM
AAD -->|"SSO/MFA"| HCM
AAD -->|"SSO/MFA"| QMS
VMW --- BACKUP
NET --- FW
%% Security 
FW -->|"logs"| SIEM
AD -->|"events"| SIEM
M365 --- DLP
AAD --- DLP
```

This integration information is loaded into Egeria as lineage relationships.

----

In [ ]:
# Show system integrations using lineage relationships

austin_system_integrations = load_csv('coco_austin_system_interactions.csv')
preview(austin_system_integrations)


In [ ]:
# Add Austin System Integrations to Egeria

token = egeria_client.create_egeria_bearer_token()

for integ in austin_system_integrations:
    interaction_id       = integ['interaction_id']
    source_system_id     = integ['source_system_id']
    source_system_name   = integ['source_system_name']
    target_system_id     = integ['target_system_id']
    target_system_name   = integ['target_system_name']
    direction            = integ['direction']
    interaction_type     = integ['interaction_type']
    protocol             = integ['protocol']
    frequency            = integ['frequency']
    data_exchanged       = integ['data_exchanged']
    description          = integ['description']

    relationship_guid=egeria_client.get_relationships_with_property_value(property_value=interaction_id,
                                                                          property_names=["label"],
                                                                          relationship_type="DataFlow")

    if relationship_guid == "No elements found":
        if direction == "one-way":
            one_way=True
        else:
            one_way=False
        
        end_1_guid=sys_map[source_system_id]
        end_2_guid=sys_map[target_system_id]
        
        body = {
            "class" : "NewRelationshipRequestBody",
            "properties": {
                "class" : "DataFlowProperties",
                "iscQualifiedName": "add qualifiedName of information supply chain here",
                "label": interaction_id,
                "oneWay" : one_way,
                "integrationStyle" : interaction_type,
                "frequency" : frequency,
                "protocol" : protocol,
                "dataExchanged": data_exchanged,
                "description": description
            }
        }
        
        print("Creating interaction " + interaction_id + " between " + end_1_guid + " and " + end_2_guid)

        relationship_guid = egeria_client.link_data_flow(element_one_guid = end_1_guid,
                                                         relationship_type_name = "DataFlow",
                                                         element_two_guid = end_2_guid,
                                                         body=body)
    
    print(f"Processed: {interaction_id}  {description[:60]}...")
    #data_flow=egeria_client.get_relationship_by_guid(guid=relationship_guid)
    #print(json.dumps(data_flow,indent=2))
    
print(f"\nDone — {len(austin_system_integrations)} Austin system interactions processed.")



-----

## Romanian Systems (EKG)

Created [ekg_systems.csv](tools/postgres-surveyor/data/ekg_systems.csv) with 30 systems. Here's a summary of what's included:

**By business unit:**
| Business Unit | Systems |
|---|---|
| IT | Active Directory, vSphere, Meraki, Veeam, Azure DevOps, Darktrace, Splunk, ServiceMgmt |
| Manufacturing | MES (Opcenter), SCADA (FactoryTalk), Cold Chain IoT |
| Quality Assurance / Control | LIMS, QMS (Veeva Vault), Chromatography (Empower), Process Modeller |
| Finance & Operations | SAP S/4HANA, Concur, PowerBI |
| Regulatory Affairs | Trackwise, Veeva Vault RIM |
| Human Resources | Workday HCM, Kronos |
| Legal | DocuSign, OpenText ECM |
| Sales / Marketing | Veeva CRM, HubSpot |
| Supply Chain | Warehouse Management System |
| Facilities | Maximo, AutoCAD Plant 3D |

**Business owners** are drawn from the employee CSV — Gheorghe Radu (Finance Director), Florin Stoica (Operations Manager), Sorin Matei (Head of Security), etc. — so the two files are consistent with each other.

----

In [ ]:
# Locate/create the Bucharest Location

token = egeria_client.create_egeria_bearer_token()

bucharest_site_id="BUC-SITE"
bucharest_site_name="Site::"+bucharest_site_id
bucharest_site_guid=egeria_client.get_element_guid_by_unique_name(bucharest_site_name)

if bucharest_site_guid == "No elements found":
    body = {
        "class" : "NewElementRequestBody",
        "properties": {
            "class" : "LocationProperties",
            "qualifiedName": bucharest_site_name,
            "identifier": bucharest_site_id,
            "displayName": "Bucharest Site",
            "description": "Former EKG Pharmaceuticals S.R.L. Location"
        },
        "initialClassifications" : {
            "FixedLocation" : {
                "class" : "FixedLocationProperties",
                "postalAddress": "Strada Fabricii nr. 14, Sector 6, București 060841, România",
                "timeZone" : "UTC+3"
            }
        }
        
    }
    bucharest_site_guid = egeria_client.create_location(body)

print ("Bucharest Site GUID: " + bucharest_site_guid)

bucharest_dc_id="BUC-DC"
bucharest_dc_name="Data centre::"+bucharest_dc_id
bucharest_dc_guid=egeria_client.get_element_guid_by_unique_name(bucharest_dc_name)

if bucharest_dc_guid == "No elements found":
    body = {
        "class" : "NewElementRequestBody",
        "properties": {
            "class" : "LocationProperties",
            "qualifiedName": bucharest_dc_name,
            "identifier": bucharest_dc_id,
            "displayName": "Bucharest Data Centre",
            "description": "Data Center at the Bucharest Site."
        },
        "initialClassifications" : {
            "FixedLocation" : {
                "class" : "FixedLocationProperties",
                "postalAddress": "Strada Fabricii nr. 14, Sector 6, București 060841, România",
                "timeZone" : "UTC+3"
            }
        },
        "parentGUID" : bucharest_site_guid,
        "parentRelationshipTypeName" : "NestedLocation",
        "parentAtEnd1" : True
    }
    bucharest_dc_guid = egeria_client.create_location(body)

print ("Bucharest Data centre GUID: " + bucharest_dc_guid)


In [ ]:
# Create the Bucharest Data Centre subsystem

token = egeria_client.create_egeria_bearer_token()

bucharest_subsystem_collection_name="Bucharest Data Centre Systems"
bucharest_subsystem_collection_desc="Systems located in the Bucharest data centre are self-contined at this moment and so are shown as a separate subsystem."
bucharest_subsystem_collection_qname="ITSubsystem::" + bucharest_subsystem_collection_name

body = {
    "class" : "NewElementRequestBody",
    "isOwnAnchor": True,
    "properties": {
        "class" : "ITSubsystemProperties",
        "qualifiedName" : bucharest_subsystem_collection_qname,
        "displayName" : bucharest_subsystem_collection_name,
        "description" : bucharest_subsystem_collection_desc,
        "category" : "Acquired Systems"
    },
    "parentGUID" : bucharest_dc_guid,
    "parentRelationshipTypeName" : "KnownLocation",
    "parentAtEnd1" : False
}


bucharest_tl_subsystem=egeria_client.get_element_by_unique_name(name=bucharest_subsystem_collection_qname)

if bucharest_tl_subsystem == "No elements found":
    bucharest_tl_subsystem_guid = egeria_client.create_collection(body=body)
    egeria_client.add_to_collection(collection_guid=it_systems_inventory_guid, element_guid=bucharest_tl_subsystem_guid)
else:
    bucharest_tl_subsystem_guid = bucharest_tl_subsystem.get("elementHeader").get("guid")
    
print ("Bucharest Top-level Subsystem GUID: " + bucharest_tl_subsystem_guid)



In [ ]:
# Load Bucharest Subsystems

bucharest_subsystems = load_csv('ekg_subsystems.csv')
preview(bucharest_subsystems)


In [ ]:
# Add Bucharest Subsystems to Egeria

token = egeria_client.create_egeria_bearer_token()

subsys_map = dict()

for sub_sys in bucharest_subsystems:
    subsystem_id          = sub_sys['subsystem_id']
    subsystem_name        = sub_sys['subsystem_name']
    subsystem_description = sub_sys['subsystem_description']
    business_owner        = sub_sys['business_owner']
    member_system_ids     = sub_sys['member_system_ids']
   
    qualified_name = "ITSubsystem::" + subsystem_id + "::" + subsystem_name

    subsystem_guid=egeria_client.get_element_guid_by_unique_name(qualified_name)

    if subsystem_guid == "No elements found":
    
        body = {
            "class" : "NewElementRequestBody",
            "isOwnAnchor": True,
            "properties": {
                "class" : "ITSubsystemProperties",
                "qualifiedName" : qualified_name,
                "identifier" : subsystem_id,
                "displayName" : subsystem_name,
                "description" : subsystem_description,
                "category" : "Acquired Systems"
            },
            "parentGUID" : austin_dc_guid,
            "parentRelationshipTypeName" : "KnownLocation",
            "parentAtEnd1" : False
        }

        subsystem_guid = egeria_client.create_collection(body=body)
        egeria_client.add_to_collection(collection_guid=bucharest_tl_subsystem_guid, element_guid=subsystem_guid)

    subsys_map[subsystem_id] = subsystem_guid
    
    print(f"Processed: {subsystem_id}  {subsystem_description[:60]}...")
    #asset=egeria_client.get_asset_by_guid(system_guid)
    #print(json.dumps(asset,indent=2))
    
print(f"\nDone — {len(bucharest_subsystems)} Bucharest subsystems processed.")



In [ ]:
# Load Bucharest Systems

bucharest_systems = load_csv('ekg_systems.csv')
preview(bucharest_systems)


In [ ]:
# Add Bucharest Systems to Egeria

token = egeria_client.create_egeria_bearer_token()

sys_map = dict()

for sys in bucharest_systems:
    system_identifier  = sys['system_identifier']
    subsystem_id       = sys['subsystem_id']
    system_name        = sys['system_name']
    system_description = sys['system_description']
    operating_system   = sys['operating_system']
    patch_level        = sys['patch_level']
    business_unit      = sys['business_unit']
    business_owner     = sys['business_owner']

    zone_membership = ["business-systems"]

    qualified_name = "SoftwareServer::" + system_identifier + "::" + system_name

    system_guid=egeria_client.get_element_guid_by_unique_name(qualified_name)

    if system_guid == "No elements found":
    
        body = {
            "class" : "NewElementRequestBody",
            "isOwnAnchor": True,
            "properties": {
                "class" : "SoftwareServerProperties",
                "qualifiedName" : qualified_name,
                "identifier" : system_identifier,
                "displayName" : system_name,
                "resourceName" : system_identifier,
                "description" : system_description,
                "deployedImplementationType" : "Unclassified Software Server",
                "category" : "Acquired Systems",
                "deploymentStatus" : "ACTIVE",
                "additionalProperties" : {
                    "operatingSystem" : operating_system,
                    "latestPatchLevel" : patch_level,
                    "businessUnit" : business_unit,
                    "businessOwner" : business_owner
                }
            },
            "initialClassifications" : {
                "ZoneMembership" : {
                    "class" : "ZoneMembershipProperties",
                    "zoneMembership" : zone_membership
                }
            },
            "parentGUID" : bucharest_dc_guid,
            "parentRelationshipTypeName" : "KnownLocation",
            "parentAtEnd1" : False
        }

        system_guid = egeria_client.create_asset(body=body)
        egeria_client.add_to_collection(subsys_map[subsystem_id], element_guid=system_guid)

    sys_map[system_identifier] = system_guid
    
    print(f"Processed: {system_identifier}  {system_description[:60]}...")
    #asset=egeria_client.get_asset_by_guid(system_guid)
    #print(json.dumps(asset,indent=2))
    
print(f"\nDone — {len(bucharest_systems)} Bucharest systems processed.")



----
### System Interactions

The diagram below shows all 30 EKG systems across eight subsystems with the main integration flows:

Manufacturing → Quality — MES pushes batch records into Veeva QMS; LIMS receives chromatography instrument data and feeds results up to QMS
Quality → Legal — QMS archives documents to OpenText ECM; Trackwise feeds regulatory submissions into Veeva RIM
Manufacturing & MES → SAP — production completion data drives inventory and cost accounting in S/4HANA
SAP at the centre of Finance — bi-directional flows to WMS (inventory), Workday (payroll), Concur (expenses), and Power BI (reporting)
Active Directory as the auth backbone — authenticates users into SAP, MES, LIMS, and VMware vSphere on-premises
Darktrace + Splunk for security — Darktrace monitors Meraki network traffic and feeds anomaly alerts into Splunk SIEM, which also collects logs from SAP, MES and Active Directory

```mermaid
flowchart TB
subgraph SHOP["🏭 Manufacturing"]
MES["Siemens Opcenter\nMES\nSYS-003"]
SCADA["Rockwell FactoryTalk\nSCADA\nSYS-015"]
COLD["Proact Cold Chain\nMonitor\nSYS-022"]
end
subgraph QUALITY["🧪 Quality & Regulatory"]
QMS["Veeva Vault\nQMS\nSYS-002"]
LIMS["LabWare\nLIMS\nSYS-004"]
CHR["Empower\nChromatography\nSYS-027"]
TRACK["Trackwise\nDigital\nSYS-005"]
RIM["Veeva Vault\nRIM\nSYS-021"]
ARIS["Aris Process\nModeller\nSYS-017"]
end
subgraph ERP["💰 Finance & Operations"]
SAP["SAP S/4HANA\nSYS-001"]
CONCUR["SAP Concur\nSYS-012"]
BI["Power BI\nSYS-014"]
end
subgraph SUPPLY["📦 Supply Chain"]
WMS["Warehouse\nMgmt WMS\nSYS-019"]
MAXIMO["Maximo Asset\nMgmt\nSYS-016"]
end
subgraph CUSTOMERS["🤝 Sales & Marketing"]
CRM["Veeva\nCRM\nSYS-006"]
MKT["HubSpot\nMarketing\nSYS-028"]
end
subgraph HR["👥 HR"]
HCM["Workday\nHCM\nSYS-013"]
KRONOS["Kronos\nWorkforce\nSYS-023"]
end
subgraph LEGAL["⚖️ Legal"]
ECM["OpenText\nECM\nSYS-024"]
DOCU["DocuSign\nSYS-020"]
end
subgraph IT["🖥 IT Infrastructure"]
AD["Active\nDirectory\nSYS-008"]
VMWARE["VMware\nvSphere\nSYS-011"]
VEEAM["Veeam\nBackup\nSYS-010"]
NET["Cisco Meraki\nNetwork\nSYS-009"]
ITSM["Serena Business\nMgr ITSM\nSYS-018"]
DEVOPS["Azure\nDevOps\nSYS-030"]
M365["Microsoft 365\nSYS-007"]
end
subgraph SEC["🔒 Security"]
DARK["Darktrace\nAI\nSYS-025"]
SIEM["Splunk\nSIEM\nSYS-026"]
end
subgraph FAC["🏗 Facilities"]
AUTOCAD["AutoCAD\nPlant 3D\nSYS-029"]
end
%% Manufacturing flows
SCADA -->|"telemetry"| MES
COLD -->|"temp alerts"| MES
MES -->|"batch records"| QMS
MES -->|"production data"| SAP
%% Quality flows
LIMS -->|"test results"| QMS
CHR -->|"instrument data"| LIMS
QMS -->|"deviations"| TRACK
QMS -->|"documents"| ECM
QMS -->|"SOPs"| ARIS
TRACK -->|"submissions"| RIM
RIM -->|"regulatory docs"| ECM
%% ERP flows
SAP <-->|"inventory"| WMS
SAP <-->|"payroll"| HCM
SAP -->|"financials"| BI
SAP -->|"expenses"| CONCUR
MES -->|"data"| BI
%% Supply chain
WMS -->|"asset data"| MAXIMO
MAXIMO -->|"costs"| SAP
%% Sales & marketing
CRM -->|"opportunities"| SAP
CRM --- MKT
%% HR
HCM --- KRONOS
%% Legal
DOCU -->|"signed docs"| ECM
%% IT infrastructure
AD -->|"auth"| VMWARE
AD -->|"auth"| MES
AD -->|"auth"| SAP
AD -->|"auth"| LIMS
VMWARE --- VEEAM
NET --- DARK
M365 -->|"collab"| ITSM
DEVOPS -->|"deployments"| VMWARE
%% Security
NET -->|"traffic"| DARK
DARK -->|"alerts"| SIEM
AD -->|"events"| SIEM
SAP -->|"logs"| SIEM
MES -->|"logs"| SIEM
%% Facilities
AUTOCAD -->|"schematics"| ECM
```

----

In [ ]:
# Show system integrations using lineage relationships

token = egeria_client.create_egeria_bearer_token()

bucharest_system_integrations = load_csv('ekg_system_interactions.csv')
preview(bucharest_system_integrations)


In [ ]:
# Add EKG System Integrations to Egeria

token = egeria_client.create_egeria_bearer_token()

for integ in bucharest_system_integrations:
    interaction_id       = integ['interaction_id']
    source_system_id     = integ['source_system_id']
    source_system_name   = integ['source_system_name']
    target_system_id     = integ['target_system_id']
    target_system_name   = integ['target_system_name']
    direction            = integ['direction']
    interaction_type     = integ['interaction_type']
    protocol             = integ['protocol']
    frequency            = integ['frequency']
    data_exchanged       = integ['data_exchanged']
    description          = integ['description']

    relationship_guid=egeria_client.get_relationships_with_property_value(property_value=interaction_id,
                                                                          property_names=["label"],
                                                                          relationship_type="DataFlow")

    if relationship_guid == "No elements found":
        
        if direction == "one-way":
            one_way=True
        else:
            one_way=False
        
        end_1_guid=sys_map[source_system_id]
        end_2_guid=sys_map[target_system_id]
        
        body = {
            "class" : "NewRelationshipRequestBody",
            "properties": {
                "class" : "DataFlowProperties",
                "iscQualifiedName": "add qualifiedName of information supply chain here",
                "label": interaction_id,
                "oneWay" : one_way,
                "integrationStyle" : interaction_type,
                "frequency" : frequency,
                "protocol" : protocol,
                "dataExchanged": data_exchanged,
                "description": description
            }
        }
        
        print("Creating interaction " + interaction_id + " between " + end_1_guid + " and " + end_2_guid)

        relationship_guid = egeria_client.link_data_flow(element_one_guid = end_1_guid,
                                                         relationship_type_name = "DataFlow",
                                                         element_two_guid = end_2_guid,
                                                         body=body)
    
    print(f"Processed: {interaction_id}  {description[:60]}...")
    #data_flow=egeria_client.get_relationship_by_guid(guid=relationship_guid)
    #print(json.dumps(data_flow,indent=2))
    
print(f"\nDone — {len(bucharest_system_integrations)} Bucharest system interactions processed.")

